In [1]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT"
target_path = r"/mnt/lustre/lalit47/down/seg/BUSI"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def collect_segmentation_pairs():
    data_by_class = defaultdict(list)

    class_names = sorted(os.listdir(source_path))

    for class_name in class_names:
        class_dir = os.path.join(source_path, class_name)

        if not os.path.isdir(class_dir):
            continue

        for file_name in os.listdir(class_dir):

            if file_name.endswith("_mask.png"):
                continue

            base_name = os.path.splitext(file_name)[0]

            image_path = os.path.join(class_dir, file_name)
            mask_path = os.path.join(class_dir, base_name + "_mask.png")

            if not os.path.exists(mask_path):
                print(f"Mask missing for: {image_path}")
                continue

            data_by_class[class_name].append(
                (image_path, mask_path, base_name, class_name)
            )

    return data_by_class


def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_segmentation_split(data, split_name):
    image_out_dir = os.path.join(target_path, split_name, "images")
    mask_out_dir = os.path.join(target_path, split_name, "masks")

    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    for image_path, mask_path, base_name, class_name in data:
        save_name = f"{class_name}_{base_name}.png"

        image_dst = os.path.join(image_out_dir, save_name)
        mask_dst = os.path.join(mask_out_dir, save_name)

        process_and_save(image_path, image_dst)
        process_and_save(mask_path, mask_dst)


data_by_class = collect_segmentation_pairs()

train_data, val_data, test_data = stratified_split(data_by_class)

save_segmentation_split(train_data, "train")
save_segmentation_split(val_data, "val")
save_segmentation_split(test_data, "test")

print("\nSegmentation dataset created successfully!")

Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (315)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (181)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (195)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (83)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (424)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (25)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (98)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (92)_mask_1.png
Mask missing for: /mnt/lustre/lalit4

In [3]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT"
target_path = r"/mnt/lustre/lalit47/down/seg/BUSI"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)

# Remove normal images for segmentation
exclude_classes = {"normal"}


def process_and_save(src, dst, is_mask=False):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    if is_mask:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_NEAREST)
    else:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)
    return True


def collect_segmentation_pairs():
    data_by_class = defaultdict(list)

    class_names = sorted(os.listdir(source_path))

    for class_name in class_names:
        class_dir = os.path.join(source_path, class_name)

        if not os.path.isdir(class_dir):
            continue

        # Skip normal class for segmentation
        if class_name.lower() in exclude_classes:
            print(f"Skipping class for segmentation: {class_name}")
            continue

        for file_name in os.listdir(class_dir):

            if file_name.endswith("_mask.png"):
                continue

            base_name = os.path.splitext(file_name)[0]

            image_path = os.path.join(class_dir, file_name)
            mask_path = os.path.join(class_dir, base_name + "_mask.png")

            if not os.path.exists(mask_path):
                print(f"Mask missing for: {image_path}")
                continue

            data_by_class[class_name].append(
                (image_path, mask_path, base_name, class_name)
            )

    return data_by_class


def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_segmentation_split(data, split_name):
    image_out_dir = os.path.join(target_path, split_name, "images")
    mask_out_dir = os.path.join(target_path, split_name, "masks")

    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    for image_path, mask_path, base_name, class_name in data:
        save_name = f"{class_name}_{base_name}.png"

        image_dst = os.path.join(image_out_dir, save_name)
        mask_dst = os.path.join(mask_out_dir, save_name)

        process_and_save(image_path, image_dst, is_mask=False)
        process_and_save(mask_path, mask_dst, is_mask=True)


data_by_class = collect_segmentation_pairs()

train_data, val_data, test_data = stratified_split(data_by_class)

save_segmentation_split(train_data, "train")
save_segmentation_split(val_data, "val")
save_segmentation_split(test_data, "test")

print("\nSegmentation dataset created successfully without normal images!")

Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (315)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (181)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (195)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (83)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (424)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (25)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (98)_mask_1.png
Mask missing for: /mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT/benign/benign (92)_mask_1.png
Mask missing for: /mnt/lustre/lalit4

In [2]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/pre_train/Dataset_BUSI/Dataset_BUSI_with_GT"
target_path = r"/mnt/lustre/lalit47/down/class/BUSI"

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def collect_classification_images():
    data_by_class = defaultdict(list)

    class_names = sorted(os.listdir(source_path))

    for class_name in class_names:
        class_dir = os.path.join(source_path, class_name)

        if not os.path.isdir(class_dir):
            continue

        for file_name in os.listdir(class_dir):

            if file_name.endswith("_mask.png"):
                continue

            if not file_name.lower().endswith((".png", ".jpg", ".jpeg")):
                continue

            image_path = os.path.join(class_dir, file_name)
            base_name = os.path.splitext(file_name)[0]

            data_by_class[class_name].append(
                (image_path, base_name, class_name)
            )

    return data_by_class


def stratified_split(data_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_name, samples in data_by_class.items():
        random.shuffle(samples)

        n = len(samples)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_samples = samples[:n_train]
        val_samples = samples[n_train:n_train + n_val]
        test_samples = samples[n_train + n_val:]

        train_data.extend(train_samples)
        val_data.extend(val_samples)
        test_data.extend(test_samples)

        print(
            f"{class_name}: total={n}, "
            f"train={len(train_samples)}, "
            f"val={len(val_samples)}, "
            f"test={len(test_samples)}"
        )

    return train_data, val_data, test_data


def save_classification_split(data, split_name):
    for image_path, base_name, class_name in data:
        class_out_dir = os.path.join(target_path, split_name, class_name)
        os.makedirs(class_out_dir, exist_ok=True)

        image_dst = os.path.join(class_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)


data_by_class = collect_classification_images()

train_data, val_data, test_data = stratified_split(data_by_class)

save_classification_split(train_data, "train")
save_classification_split(val_data, "val")
save_classification_split(test_data, "test")

print("\nClassification dataset created successfully!")

benign: total=454, train=317, val=45, test=92
malignant: total=211, train=147, val=21, test=43
normal: total=133, train=93, val=13, test=27

Classification dataset created successfully!


In [16]:
from pathlib import Path
import numpy as np
from PIL import Image


def check_mask_min_max(dataset_root, num_classes=2):
    dataset_root = Path(dataset_root)
    splits = ["train", "val", "test"]

    expected_min = 0
    expected_max = num_classes - 1

    print(f"Expected mask value range: min={expected_min}, max={expected_max}")
    print("=" * 60)

    all_values_min = []
    all_values_max = []
    bad_files = []

    for split in splits:
        mask_dir = dataset_root / split / "masks"

        if not mask_dir.exists():
            print(f"[WARNING] Missing folder: {mask_dir}")
            continue

        mask_files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]:
            mask_files.extend(mask_dir.glob(ext))

        mask_files = sorted(mask_files)

        if not mask_files:
            print(f"[WARNING] No mask files found in {mask_dir}")
            continue

        split_min = float("inf")
        split_max = float("-inf")

        print(f"\nChecking {split}: {len(mask_files)} masks")

        for mask_path in mask_files:
            mask = np.array(Image.open(mask_path))

            # If RGB mask, this is usually not correct for class-index masks
            if mask.ndim == 3:
                bad_files.append((split, mask_path.name, "RGB mask"))
                continue

            mask_min = int(mask.min())
            mask_max = int(mask.max())

            split_min = min(split_min, mask_min)
            split_max = max(split_max, mask_max)

            all_values_min.append(mask_min)
            all_values_max.append(mask_max)

            if mask_min < expected_min or mask_max > expected_max:
                bad_files.append((split, mask_path.name, f"min={mask_min}, max={mask_max}"))

        print(f"{split} mask range: min={split_min}, max={split_max}")

    print("\n" + "=" * 60)
    print("FINAL REPORT")
    print("=" * 60)

    if all_values_min and all_values_max:
        dataset_min = min(all_values_min)
        dataset_max = max(all_values_max)

        print(f"Dataset mask range: min={dataset_min}, max={dataset_max}")

        if dataset_min == expected_min and dataset_max == expected_max and not bad_files:
            print("All masks are correct.")
        elif dataset_min >= expected_min and dataset_max <= expected_max:
            print("All mask values are inside the expected range.")
            print("But not every split/file may contain all classes.")
        else:
            print("Some masks have values outside the expected range.")

    if bad_files:
        print("\nProblem files:")
        for split, name, issue in bad_files[:50]:
            print(f"{split}: {name} -> {issue}")

        if len(bad_files) > 50:
            print(f"... and {len(bad_files) - 50} more problem files.")


# Example 1: binary segmentation, labels should be 0 and 1
check_mask_min_max(
    dataset_root="/mnt/lustre/lalit47/down/seg/Neck Nerve",
    num_classes=2
)

# Example 2: multiclass segmentation with 5 classes, labels should be 0,1,2,3,4
# check_mask_min_max(
#     dataset_root="/path/to/your/dataset",
#     num_classes=5
# )

Expected mask value range: min=0, max=1

Checking train: 5071 masks
train mask range: min=0, max=255

Checking val: 564 masks
val mask range: min=0, max=255
[WARNING] Missing folder: /mnt/lustre/lalit47/down/seg/Neck Nerve/test/masks

FINAL REPORT
Dataset mask range: min=0, max=255
Some masks have values outside the expected range.

Problem files:
train: 10_1.png -> min=0, max=255
train: 10_100.png -> min=0, max=255
train: 10_104.png -> min=0, max=255
train: 10_109.png -> min=0, max=255
train: 10_11.png -> min=0, max=255
train: 10_15.png -> min=0, max=255
train: 10_16.png -> min=0, max=255
train: 10_18.png -> min=0, max=255
train: 10_2.png -> min=0, max=255
train: 10_24.png -> min=0, max=255
train: 10_26.png -> min=0, max=255
train: 10_27.png -> min=0, max=255
train: 10_28.png -> min=0, max=255
train: 10_3.png -> min=0, max=255
train: 10_30.png -> min=0, max=255
train: 10_32.png -> min=0, max=255
train: 10_35.png -> min=0, max=255
train: 10_36.png -> min=0, max=255
train: 10_37.png -> 

In [ ]:
from pathlib import Path
import shutil
import numpy as np
from PIL import Image


def convert_masks_and_copy_images(dataset_root, output_root):
    dataset_root = Path(dataset_root)
    output_root = Path(output_root)

    splits = ["train", "val", "test"]

    image_extensions = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]
    mask_extensions = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]

    for split in splits:
        input_image_dir = dataset_root / split / "images"
        input_mask_dir = dataset_root / split / "masks"

        output_image_dir = output_root / split / "images"
        output_mask_dir = output_root / split / "masks"

        output_image_dir.mkdir(parents=True, exist_ok=True)
        output_mask_dir.mkdir(parents=True, exist_ok=True)

        # -------------------------
        # Copy images
        # -------------------------
        if input_image_dir.exists():
            image_files = []
            for ext in image_extensions:
                image_files.extend(input_image_dir.glob(ext))

            image_files = sorted(image_files)

            print(f"\nCopying {split} images: {len(image_files)} files")

            for image_path in image_files:
                output_image_path = output_image_dir / image_path.name
                shutil.copy2(image_path, output_image_path)
        else:
            print(f"[WARNING] Missing image folder: {input_image_dir}")

        # -------------------------
        # Convert masks to 0/255
        # -------------------------
        if not input_mask_dir.exists():
            print(f"[WARNING] Missing mask folder: {input_mask_dir}")
            continue

        mask_files = []
        for ext in mask_extensions:
            mask_files.extend(input_mask_dir.glob(ext))

        mask_files = sorted(mask_files)

        print(f"Converting {split} masks: {len(mask_files)} files")

        for mask_path in mask_files:
            mask = np.array(Image.open(mask_path))

            # If RGB/RGBA mask, take first channel
            if mask.ndim == 3:
                mask = mask[:, :, 0]

            original_unique = np.unique(mask)

            # 0 stays 0, every non-zero value becomes 255
            binary_mask = np.where(mask > 0, 255, 0).astype(np.uint8)

            new_unique = np.unique(binary_mask)

            # Save masks as PNG to avoid JPEG label corruption
            output_mask_path = output_mask_dir / mask_path.with_suffix(".png").name
            Image.fromarray(binary_mask, mode="L").save(output_mask_path)

            print(
                f"{split}: {mask_path.name} "
                f"{original_unique.tolist()} -> {new_unique.tolist()}"
            )

    print("\nDone.")
    print(f"Converted dataset saved to: {output_root}")


def verify_converted_dataset(dataset_root):
    dataset_root = Path(dataset_root)

    splits = ["train", "val", "test"]
    extensions = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]

    print("\nVerifying converted dataset")
    print("=" * 60)

    for split in splits:
        image_dir = dataset_root / split / "images"
        mask_dir = dataset_root / split / "masks"

        image_files = []
        mask_files = []

        if image_dir.exists():
            for ext in extensions:
                image_files.extend(image_dir.glob(ext))

        if mask_dir.exists():
            for ext in extensions:
                mask_files.extend(mask_dir.glob(ext))

        image_files = sorted(image_files)
        mask_files = sorted(mask_files)

        print(f"\n{split}")
        print(f"Images: {len(image_files)}")
        print(f"Masks:  {len(mask_files)}")

        bad_masks = []

        for mask_path in mask_files:
            mask = np.array(Image.open(mask_path))

            if mask.ndim == 3:
                mask = mask[:, :, 0]

            unique_values = np.unique(mask)

            if not set(unique_values.tolist()).issubset({0, 255}):
                bad_masks.append((mask_path.name, unique_values.tolist()))

        if len(bad_masks) == 0:
            print("Masks are correct: only 0 and 255")
        else:
            print("Bad masks found:")
            for name, values in bad_masks[:20]:
                print(f"{name}: {values}")

        image_names = {p.stem for p in image_files}
        mask_names = {p.stem for p in mask_files}

        missing_masks = sorted(image_names - mask_names)
        missing_images = sorted(mask_names - image_names)

        if not missing_masks and not missing_images:
            print("Image-mask filenames match.")
        else:
            if missing_masks:
                print(f"Images without masks: {missing_masks[:20]}")
            if missing_images:
                print(f"Masks without images: {missing_images[:20]}")




convert_masks_and_copy_images(
    dataset_root="/mnt/lustre/lalit47/down/seg/MMOTU",
    output_root="/mnt/lustre/lalit47/down/seg/MMOTU_converted"
)

verify_converted_dataset(
    dataset_root="/mnt/lustre/lalit47/down/seg/MMOTU_converted"
)

In [ ]:
from pathlib import Path
import shutil
import numpy as np
from PIL import Image


def convert_masks_copy_images_skip_empty(dataset_root, output_root):
    dataset_root = Path(dataset_root)
    output_root = Path(output_root)

    splits = ["train", "val", "test"]

    image_extensions = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]
    mask_extensions = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]

    for split in splits:
        input_image_dir = dataset_root / split / "images"
        input_mask_dir = dataset_root / split / "masks"

        output_image_dir = output_root / split / "images"
        output_mask_dir = output_root / split / "masks"

        output_image_dir.mkdir(parents=True, exist_ok=True)
        output_mask_dir.mkdir(parents=True, exist_ok=True)

        if not input_mask_dir.exists():
            print(f"[WARNING] Missing mask folder: {input_mask_dir}")
            continue

        if not input_image_dir.exists():
            print(f"[WARNING] Missing image folder: {input_image_dir}")
            continue

        mask_files = []
        for ext in mask_extensions:
            mask_files.extend(input_mask_dir.glob(ext))

        mask_files = sorted(mask_files)

        copied_count = 0
        skipped_empty_count = 0
        missing_image_count = 0

        print(f"\nProcessing {split}: {len(mask_files)} masks")

        for mask_path in mask_files:
            mask = np.array(Image.open(mask_path))

            # If RGB/RGBA mask, take first channel
            if mask.ndim == 3:
                mask = mask[:, :, 0]

            # Convert to binary 0/255
            binary_mask = np.where(mask > 0, 255, 0).astype(np.uint8)

            # Skip if mask contains only background
            if np.max(binary_mask) == 0:
                skipped_empty_count += 1
                print(f"SKIP empty mask: {split}/{mask_path.name}")
                continue

            # Find corresponding image with same stem
            image_path = None
            for img_ext in image_extensions:
                candidate = input_image_dir / f"{mask_path.stem}{img_ext}"
                if candidate.exists():
                    image_path = candidate
                    break

            if image_path is None:
                missing_image_count += 1
                print(f"[WARNING] No image found for mask: {split}/{mask_path.name}")
                continue

            # Save converted mask as PNG
            output_mask_path = output_mask_dir / f"{mask_path.stem}.png"
            Image.fromarray(binary_mask, mode="L").save(output_mask_path)

            # Copy corresponding image
            output_image_path = output_image_dir / image_path.name
            shutil.copy2(image_path, output_image_path)

            copied_count += 1

        print(f"{split} summary")
        print(f"Copied image-mask pairs: {copied_count}")
        print(f"Skipped empty masks: {skipped_empty_count}")
        print(f"Missing images: {missing_image_count}")

    print("\nDone.")
    print(f"Filtered converted dataset saved to: {output_root}")


def verify_no_empty_masks(dataset_root):
    dataset_root = Path(dataset_root)

    splits = ["train", "val", "test"]
    extensions = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp"]

    print("\nVerifying filtered dataset")
    print("=" * 60)

    for split in splits:
        image_dir = dataset_root / split / "images"
        mask_dir = dataset_root / split / "masks"

        image_files = []
        mask_files = []

        if image_dir.exists():
            for ext in extensions:
                image_files.extend(image_dir.glob(ext))

        if mask_dir.exists():
            for ext in extensions:
                mask_files.extend(mask_dir.glob(ext))

        image_files = sorted(image_files)
        mask_files = sorted(mask_files)

        empty_masks = []
        bad_value_masks = []

        for mask_path in mask_files:
            mask = np.array(Image.open(mask_path))

            if mask.ndim == 3:
                mask = mask[:, :, 0]

            unique_values = np.unique(mask)

            if set(unique_values.tolist()) == {0}:
                empty_masks.append(mask_path.name)

            if not set(unique_values.tolist()).issubset({0, 255}):
                bad_value_masks.append((mask_path.name, unique_values.tolist()))

        image_names = {p.stem for p in image_files}
        mask_names = {p.stem for p in mask_files}

        missing_masks = sorted(image_names - mask_names)
        missing_images = sorted(mask_names - image_names)

        print(f"\n{split}")
        print(f"Images: {len(image_files)}")
        print(f"Masks:  {len(mask_files)}")

        if len(empty_masks) == 0:
            print("No empty masks found.")
        else:
            print(f"Empty masks found: {empty_masks[:20]}")

        if len(bad_value_masks) == 0:
            print("All masks contain only 0 and 255.")
        else:
            print("Bad mask values found:")
            for name, values in bad_value_masks[:20]:
                print(f"{name}: {values}")

        if not missing_masks and not missing_images:
            print("Image-mask filenames match.")
        else:
            if missing_masks:
                print(f"Images without masks: {missing_masks[:20]}")
            if missing_images:
                print(f"Masks without images: {missing_images[:20]}")


convert_masks_copy_images_skip_empty(
    dataset_root="/mnt/lustre/lalit47/down/seg/Neck Nerve",
    output_root="/mnt/lustre/lalit47/down/seg/Neck_Nerve_Converted"
)

verify_no_empty_masks(
    dataset_root="/mnt/lustre/lalit47/down/seg/Neck_Nerve_Converted"
)